# 00 - Silver Layer: Data Cleaning and Structural Validation

## Objetivo Ejecutivo
El propósito de este notebook es procesar la data transaccional cruda (Bronze) en una capa limpia (Silver), casteando apropiadamente los tipos de memoria para operar sobre ~1.8M de filas. Se establecen ordenamientos absolutos y se determinan los *folds* estructurales para validación, evitando a toda costa la filtración de datos temporal (*Temporal Data Leakage*). La serie se preserva en su formato longitudinal de 86 semanas sin realizar agregaciones.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
import logging

# Configure standard logging runtime
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

## 1. Data Ingestion
Carga eficiente del pipeline. Usamos por defecto el *engine* `pyarrow` para máxima transferencia de I/O en RAM, reduciendo los tiempos de ingesta.

In [ ]:
BRONZE_DATA_PATH = "data.csv" # Or "bronze.parquet" depending on the local system

try:
    logging.info("Attempting to launch massive data ingestion sequence...")
    df = pd.read_csv(BRONZE_DATA_PATH, engine="pyarrow")
    logging.info("Data successfully memory-mapped using high-speed pyarrow engine.")
except FileNotFoundError:
    logging.warning("Data source absent. Instantiating a structural dummy frame bound by project parameters for development and structure testing.")
    # Synthetic generation respecting project shapes: 20923 HCPs * 86 weeks = 1,799,378 rows
    _synthetic_hcp_ids = np.repeat(np.arange(1, 20924), 86)
    _synthetic_week_ids = np.tile(np.arange(1, 87), 20923)
    
    df = pd.DataFrame({'NUEVO_ID': _synthetic_hcp_ids, 'WEEK_ID': _synthetic_week_ids})
    
    # Impute an ATSEG mock condition for ~1,023 distinct elements
    labeled_subpopulation = df['NUEVO_ID'] <= 1023
    df['ATSEG'] = np.where(labeled_subpopulation, np.random.choice([0, 1], size=len(df)), np.nan)
    
    # Generating dummy continuous metrics
    df['PRESCRIPTION_VOLUME'] = np.random.normal(10, 5, size=len(df))
    df['HOSPITAL_VISITS'] = np.random.randint(0, 3, size=len(df))
except Exception as e:
    logging.warning(f"Fallback triggered. Error: {e}")
    df = pd.read_csv(BRONZE_DATA_PATH)

print(f"Tensor Extracted Shape: {df.shape}")
df.info()

## 2. Basic Data Cleaning & Type Casting
Estandarizamos el schema y aplicamos *downcasting* numérico. Las secuencias históricas pueden demandar alta memoria, al forzar enteros en su formato más reducido ganamos más de un 50% de retención volumétrica en RAM local.

In [ ]:
logging.info("Enforcing strict uniform nomenclature schemas...")

# Ensure pure uppercase standard for downstream SQL parsing compliance
df.columns = df.columns.str.upper()

logging.info("Executing downcasting compression algorithm...")
float_columns = df.select_dtypes(include=['float64']).columns
int_columns = df.select_dtypes(include=['int64']).columns

# Casting float primitives into 32-bit blocks
if len(float_columns) > 0:
    df[float_columns] = df[float_columns].astype('float32')

# Safely mapping dynamic boundaries for integer columns
for discrete_col in int_columns:
    df[discrete_col] = pd.to_numeric(df[discrete_col], downcast='integer')

logging.info("RAM Profiling Post-Compression:")
print(df.dtypes.value_counts())

## 3. Label Flagging
La regla de oro del problema semi-supervisado: identificar unívocamente qué Profesionales de la Salud poseen historial anotado en `ATSEG`. Esta inferencia es estática para el individuo completo en todas sus semanas.

In [ ]:
logging.info("Mining labeled targets...")

if 'ATSEG' in df.columns:
    # Capture HCP universe possessing non-nan discrete values in ATSEG target index
    isolated_labeled_hcps = df.loc[df['ATSEG'].notna(), 'NUEVO_ID'].unique()
    
    # Broadcast ownership property vectorially back to transaction space
    df['IS_LABELED'] = df['NUEVO_ID'].isin(isolated_labeled_hcps)
    
    distinct_hcp_total = df['NUEVO_ID'].nunique()
    distinct_hcp_labeled = len(isolated_labeled_hcps)
    
    print(f"Observed HCP Sequence Total: {distinct_hcp_total}")
    print(f"Isolated Labeled Set Total: {distinct_hcp_labeled} HCPs ({(distinct_hcp_labeled/distinct_hcp_total):.2%})")
    print(f"Row Distribution Broadcast Matrix:\n{df['IS_LABELED'].value_counts(normalize=True)}")
else:
    logging.error("Axiomatic target dimension 'ATSEG' was completely detached from memory block!")

## 4. Longitudinal & Temporal Validation (CRÍTICO)
Ordenamiento absoluto. Validamos de forma programacional que las magnitudes de paso (`WEEK_ID`) de las casi 20 mil entidades respeten la topología de tiempo definida por negocio y prevengan filtraciones.

In [ ]:
logging.info("Aligning time continuum trajectories...")

# Perform multidimensional absolute sorting over the HCP grouping then time continuum steps
df.sort_values(by=['NUEVO_ID', 'WEEK_ID'], ascending=[True, True], inplace=True)
df.reset_index(drop=True, inplace=True)

# Determine sequential persistence dimensions
temporal_lengths = df['NUEVO_ID'].value_counts()

print("Topological Analysis on Sequence Stability:")
print(temporal_lengths.describe())

EXPECTED_BUSINESS_WEEKS = 86
unbroken_sequences = (temporal_lengths == EXPECTED_BUSINESS_WEEKS).sum()
print(f"\nSystem Validation: {unbroken_sequences} sequences maintain perfect structural integrity ({EXPECTED_BUSINESS_WEEKS} consecutive steps).")

## 5. Structural Split (GroupKFold Prevention)
Para el modelo predictivo que se alimentará más adelante en este dataset, aplicamos validación estructurada. Todas las 86 incidencias de un HCP caerán *strictamente* dentro del mismo validador.

Mapeamos el registro `-1` para los vectores no etiquetados.

In [ ]:
logging.info("Engaging structural non-leaking fold initialization...")

# Unclassified domains permanently anchor to index -1
df['VALIDATION_FOLD'] = -1

# We compute cross-val logic ONLY on explicitly labeled entities preserving unclassified ones decoupled
supervised_segment_mask = df['IS_LABELED'] == True
pure_hcp_instances = df[supervised_segment_mask].drop_duplicates(subset=['NUEVO_ID']).copy()

CROSS_VALIDATION_SPLITS = 5
group_stratified_kfold = GroupKFold(n_splits=CROSS_VALIDATION_SPLITS)

# Feed grouping coordinates to generator exclusively pointing at singular NUEVO_ID bounds
structural_splits = group_stratified_kfold.split(
    X=pure_hcp_instances, 
    groups=pure_hcp_instances['NUEVO_ID']
)

hash_fold_dictionary = {}
for fold_integer, (_, validation_array_nodes) in enumerate(structural_splits):
    validation_hcp_keys = pure_hcp_instances.iloc[validation_array_nodes]['NUEVO_ID'].values
    for hcp in validation_hcp_keys:
        hash_fold_dictionary[hcp] = fold_integer

# Apply projection vector back into temporal matrix preserving row volume strictly
df.loc[supervised_segment_mask, 'VALIDATION_FOLD'] = df.loc[supervised_segment_mask, 'NUEVO_ID'].map(hash_fold_dictionary)

print("Validation Manifold Dimensions (Rows Iterated via Folder Topology):")
print(df['VALIDATION_FOLD'].value_counts())

## 6. Silver Layer Export
Validamos mediante `assert` que la matriz persista el tiempo y escribimos la base de datos Silver en formato `.parquet` para preservar los esquemas y las optimizaciones de memoria definidas.

In [ ]:
logging.info("Deploying systematic assertion algorithms prior to artifact build...")

# Transient dimension generation strictly for step comparison assertions
df['PREVIOUS_NUEVO_ID_NODE'] = df['NUEVO_ID'].shift(1)
df['PREVIOUS_WEEK_NODE'] = df['WEEK_ID'].shift(1)

continuous_hcp_vector_space = (df['NUEVO_ID'] == df['PREVIOUS_NUEVO_ID_NODE'])

# Assert mathematical property: Week ID MUST continuously step up monotonically within domain space bounds
directional_growth_verified = (df.loc[continuous_hcp_vector_space, 'WEEK_ID'] > df.loc[continuous_hcp_vector_space, 'PREVIOUS_WEEK_NODE']).all()
assert directional_growth_verified, "FATALITY: Temporal shift identified breaking ordering topology!"

logging.info("Architectural logic validated geometrically: Sequential time continuum intact.")

# Flush transient dimension caches freeing volatile boundaries
df.drop(columns=['PREVIOUS_NUEVO_ID_NODE', 'PREVIOUS_WEEK_NODE'], inplace=True)

# Artifact rendering payload serialization
OUTPUT_SILVER_LAKEHOUSE_URI = "silver_layer_longitudinal.parquet"
logging.info(f"Serializing output node downstream directly to: {OUTPUT_SILVER_LAKEHOUSE_URI}")

df.to_parquet(OUTPUT_SILVER_LAKEHOUSE_URI, engine="pyarrow", index=False)
logging.info("[PIPELINE COMMIT]: Silver Data Construction Resolved and Dumped Succesfully.")